In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory


import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/shrutirao12/frontend/empathetic_frontend/README.md
/kaggle/input/datasets/shrutirao12/frontend/empathetic_frontend/app.py
/kaggle/input/datasets/shrutirao12/frontend/empathetic_frontend/requirements.txt
/kaggle/input/datasets/shrutirao12/frontend/empathetic_frontend/templates/index.html
/kaggle/input/datasets/shrutirao12/newwproj/final_ready/train.py
/kaggle/input/datasets/shrutirao12/newwproj/final_ready/inference.py
/kaggle/input/datasets/shrutirao12/newwproj/final_ready/model.py
/kaggle/input/datasets/shrutirao12/newwproj/final_ready/config.py
/kaggle/input/datasets/shrutirao12/newwproj/final_ready/dataset.py
/kaggle/input/notebooks/shrutirao12/new-proj/train.py
/kaggle/input/notebooks/shrutirao12/new-proj/inference.py
/kaggle/input/notebooks/shrutirao12/new-proj/__results__.html
/kaggle/input/notebooks/shrutirao12/new-proj/model.py
/kaggle/input/notebooks/shrutirao12/new-proj/config.py
/kaggle/input/notebooks/shrutirao12/new-proj/__huggingface_repos__.json
/k

In [2]:
!pip install "datasets<3.0.0" sacrebleu -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 11.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 14.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2024.6.1 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [3]:
import subprocess
subprocess.run(["pip", "install", "sacrebleu", "-q"])

import sys, os

CODE_DIR  = "/kaggle/input/datasets/shrutirao12/newwproj/final_ready"
CKPT_PATH = "/kaggle/input/notebooks/shrutirao12/new-proj/checkpoints/best_decoupled_model.pt"

sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)

print("Checkpoint exists:", os.path.exists(CKPT_PATH))

Checkpoint exists: True


In [4]:
from inference import load_model
from config import DEVICE
import time

print(f"Device set to: {DEVICE}")

print("[1/3] Downloading/Loading Base HuggingFace Models...")
start = time.time()
# This is where it connects to the internet! If it hangs here, it's a network issue.
from model import DecoupledEmpatheticModel
from config import ROBERTA_MODEL_NAME, BART_MODEL_NAME, NUM_EMOTIONS
model = DecoupledEmpatheticModel(ROBERTA_MODEL_NAME, BART_MODEL_NAME, NUM_EMOTIONS)
print(f"  -> Done in {time.time() - start:.1f}s")


print("[2/3] Fetching Tokenizers...")
start = time.time()
from transformers import RobertaTokenizer, BartTokenizer
rob_tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")
print(f"  -> Done in {time.time() - start:.1f}s")


print("[3/3] Loading your .pt Checkpoint weights into the model...")
import torch
CKPT_PATH = "/kaggle/input/notebooks/shrutirao12/new-proj/checkpoints/best_decoupled_model.pt"

start = time.time()
model.load_state_dict(torch.load(CKPT_PATH, weights_only=True, map_location=DEVICE))
model.to(DEVICE)
model.eval()
print(f"  -> Done in {time.time() - start:.1f}s")

print("\nSuccess! Model is fully loaded.")


Device set to: cuda
[1/3] Downloading/Loading Base HuggingFace Models...


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

tokenizer.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  -> Done in 2.0s
[3/3] Loading your .pt Checkpoint weights into the model...
  -> Done in 10.2s

Success! Model is fully loaded.


In [5]:
# Patch config threshold live — no retraining needed
import config
config.CONFIDENCE_THRESHOLD = 0.30
print("Threshold updated to 0.30")

Threshold updated to 0.30


In [6]:
# ── DIAGNOSTIC: Check what Phase 1 actually learned ──────────────────────────
import torch
from datasets import load_dataset
from tqdm import tqdm
import sys, os

CODE_DIR = "/kaggle/input/datasets/shrutirao12/newwproj/final_ready"
CKPT_PATH = "/kaggle/input/notebooks/shrutirao12/new-proj/checkpoints/best_roberta_head.pt"

sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)

from config import DEVICE, ROBERTA_MODEL_NAME, NUM_EMOTIONS, MAX_LEN
from model import DecoupledEmpatheticModel

# Load only the RoBERTa + classifier
diag_model = DecoupledEmpatheticModel(ROBERTA_MODEL_NAME, "facebook/bart-base", NUM_EMOTIONS)
ckpt = torch.load(CKPT_PATH, weights_only=True, map_location=DEVICE)
diag_model.roberta.load_state_dict(ckpt['roberta'])
diag_model.classifier.load_state_dict(ckpt['classifier'])
diag_model.to(DEVICE)
diag_model.eval()

from transformers import RobertaTokenizer
rob_tok = RobertaTokenizer.from_pretrained("roberta-base")

# Test on 5 obvious inputs and print raw probability distributions
test_inputs = [
    ("I just found out my dog passed away this morning",   "grief/sadness"),
    ("I just got accepted into my dream college!",          "excitement/joy"),
    ("My roommate keeps eating my food and I am furious",   "anger/annoyance"),
    ("I feel so lonely, nobody talks to me",                "sadness/loneliness"),
    ("I went to the grocery store today",                   "neutral"),
]

from config import GO_EMOTIONS
print(f"\n{'='*60}")
print("Phase 1 classifier diagnostic")
print(f"{'='*60}")

with torch.no_grad():
    for text, expected in test_inputs:
        enc = rob_tok(text, max_length=MAX_LEN, truncation=True, return_tensors="pt").to(DEVICE)
        out = diag_model.forward_phase1(enc['input_ids'], enc['attention_mask'])
        probs = torch.softmax(out['logits'], dim=-1)[0]
        top5 = torch.topk(probs, 5)
        
        print(f"\nInput   : {text}")
        print(f"Expected: {expected}")
        print(f"Top 5 predictions:")
        for score, idx in zip(top5.values, top5.indices):
            print(f"  {GO_EMOTIONS[idx]:<20} {score.item():.3f}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]


Phase 1 classifier diagnostic

Input   : I just found out my dog passed away this morning
Expected: grief/sadness
Top 5 predictions:
  grief                0.274
  sadness              0.232
  neutral              0.123
  surprise             0.089
  realization          0.061

Input   : I just got accepted into my dream college!
Expected: excitement/joy
Top 5 predictions:
  excitement           0.471
  approval             0.182
  joy                  0.124
  neutral              0.089
  admiration           0.047

Input   : My roommate keeps eating my food and I am furious
Expected: anger/annoyance
Top 5 predictions:
  anger                0.880
  neutral              0.061
  annoyance            0.035
  disgust              0.007
  disappointment       0.004

Input   : I feel so lonely, nobody talks to me
Expected: sadness/loneliness
Top 5 predictions:
  sadness              0.568
  disappointment       0.311
  neutral              0.023
  annoyance            0.020
  grief        

In [7]:
!pip install flask pyngrok -q

In [31]:
app_code = '''
import sys, os, torch
sys.path.insert(0, "/kaggle/input/datasets/shrutirao12/newwproj/final_ready")
os.chdir("/kaggle/input/datasets/shrutirao12/newwproj/final_ready")

from flask import Flask, render_template_string, request, jsonify, session
from transformers.modeling_outputs import BaseModelOutput
from config import MAX_LEN, DEVICE, LABEL_TO_EMOTION, CONFIDENCE_THRESHOLD, NEUTRAL_IDX, GO_EMOTIONS, ROBERTA_MODEL_NAME, BART_MODEL_NAME, NUM_EMOTIONS
from model import DecoupledEmpatheticModel
from dataset import get_tokenizers

app = Flask(__name__)
app.secret_key = "emp_secret_999"

EMOTION_MERGE = {
    "grief":"sadness","remorse":"sadness","disappointment":"sadness",
    "annoyance":"anger","disgust":"anger","disapproval":"anger",
    "nervousness":"fear","realization":"surprise",
    "relief":"optimism","desire":"optimism"
}

RESPONSE_TONE_MAP = {
    "sadness":      "caring",
    "anger":        "caring",
    "fear":         "caring",
    "embarrassment":"caring",
    "confusion":    "curious",
    "optimism":     "hopeful",
    "excitement":   "excited",
    "joy":          "joyful",
    "surprise":     "surprised",
    "pride":        "impressed",
    "admiration":   "impressed",
    "gratitude":    "grateful",
    "love":         "sentimental",
    "curiosity":    "curious",
    "neutral":      "caring",
}

COLORS = {
    "sadness":"#5b8dee","grief":"#5b8dee","fear":"#7c5cbf",
    "anger":"#e05555","annoyance":"#e07a55","excitement":"#e8a838",
    "joy":"#f0c040","admiration":"#e8a838","surprise":"#38c4a8",
    "curiosity":"#38a8c4","confusion":"#8888cc","love":"#e05580",
    "gratitude":"#55c460","optimism":"#66bb6a","neutral":"#888899","caring":"#55b4c4",
}

model = None
rob_tokenizer = None
bart_tokenizer = None
bad_words_ids = None

def load():
    global model, rob_tokenizer, bart_tokenizer, bad_words_ids
    CKPT = "/kaggle/input/notebooks/shrutirao12/new-proj/checkpoints/best_decoupled_model.pt"
    rob_tokenizer, bart_tokenizer = get_tokenizers()
    model = DecoupledEmpatheticModel(ROBERTA_MODEL_NAME, BART_MODEL_NAME, NUM_EMOTIONS)
    model.load_state_dict(torch.load(CKPT, weights_only=True, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()

    forbidden_words = [
        "I", " I", "i", " i", "I\'m", " I\'m", "i\'m", " i\'m",
        "me", " me", "my", " my", "mine", " mine", "myself", " myself",
        "I\'ll", " I\'ll", "i\'ll", " i\'ll", "I\'ve", " I\'ve", "i\'ve", " i\'ve",
        "We", " We", "we", " we", "Us", " Us", "us", " us",
        "I\'d", " I\'d", "i\'d", " i\'d"
    ]
    bad_words_ids = []
    for word in forbidden_words:
        ids = bart_tokenizer(word, add_special_tokens=False).input_ids
        bad_words_ids.append(ids)
    print("Model loaded.")

HTML = open("/kaggle/working/index.html").read()

@app.route("/")
def index():
    session["history"] = []
    return render_template_string(HTML)

@app.route("/chat", methods=["POST"])
def chat():
    user_input = request.json.get("message", "").strip()
    if not user_input:
        return jsonify({"error": "empty"}), 400

    history = session.get("history", [])

    with torch.no_grad():

        # Step 1: Detect emotion via RoBERTa
        rob_inputs = rob_tokenizer(
            user_input, max_length=MAX_LEN, truncation=True, return_tensors="pt"
        ).to(DEVICE)
        rob_out = model.roberta(
            input_ids=rob_inputs["input_ids"],
            attention_mask=rob_inputs["attention_mask"]
        )
        cls   = rob_out.last_hidden_state[:, 0, :]
        probs = torch.softmax(model.classifier(cls), dim=-1)
        max_prob, pred_idx = torch.max(probs, dim=-1)

        prob        = max_prob.item()
        raw_emotion = LABEL_TO_EMOTION[pred_idx.item()]
        emotion     = EMOTION_MERGE.get(raw_emotion, raw_emotion)

        is_fallback = prob < CONFIDENCE_THRESHOLD
        if is_fallback:
            emotion = "neutral"
            probs[0] = 0.0
            probs[0, GO_EMOTIONS.index("neutral")] = 1.0
            display = f"neutral (low conf. {prob:.2f})"
        else:
            display = raw_emotion if raw_emotion == emotion else f"{raw_emotion} -> {emotion}"

        # Step 2: Tone map — IDENTICAL to inference cell
        response_emotion = RESPONSE_TONE_MAP.get(emotion, "caring")

        # Step 3: Build context — IDENTICAL order to inference cell
        # append user FIRST, then build history string, exactly like custom_chat
        history.append(f"User: {user_input}")
        history_context = " </s> ".join(history[-6:])
        full_context = f"[{response_emotion}] {history_context}"

        bart_inputs = bart_tokenizer(
            full_context, max_length=MAX_LEN, truncation=True, return_tensors="pt"
        ).to(DEVICE)

        # Step 4: Fusion & Generation — IDENTICAL to inference cell
        e_emotion = torch.matmul(probs, model.emotion_embeddings.weight).unsqueeze(1)
        enc_out = model.bart.model.encoder(
            input_ids=bart_inputs["input_ids"],
            attention_mask=bart_inputs["attention_mask"],
            return_dict=True
        )
        H_fused = model.fusion(enc_out.last_hidden_state, e_emotion)
        encoder_outputs = BaseModelOutput(
            last_hidden_state=H_fused,
            hidden_states=enc_out.hidden_states,
            attentions=enc_out.attentions
        )

        gen_ids = model.bart.generate(
            encoder_outputs=encoder_outputs,
            attention_mask=bart_inputs["attention_mask"],
            max_new_tokens=40,
            min_length=10,
            no_repeat_ngram_size=3,
            do_sample=True,
            top_p=0.92,
            temperature=0.80,
            num_beams=3,
            num_return_sequences=1,
            bad_words_ids=bad_words_ids,
        )
        response = bart_tokenizer.decode(gen_ids[0], skip_special_tokens=True).strip()

    # append bot AFTER generation, exactly like custom_chat
    history.append(f"Bot: {response}")
    if len(history) > 12:
        history = history[-12:]
    session["history"] = history

    return jsonify({
        "response":      response,
        "emotion":       display,
        "emotion_raw":   emotion,
        "emotion_color": COLORS.get(emotion, "#888899"),
        "confidence":    int(prob * 100),
        "is_fallback":   is_fallback,
    })

@app.route("/clear", methods=["POST"])
def clear():
    session["history"] = []
    return jsonify({"status": "cleared"})

@app.route("/status")
def status():
    return jsonify({"model_loaded": model is not None, "device": str(DEVICE)})

if __name__ == "__main__":
    load()
    app.run(host="0.0.0.0", port=5000, debug=False)
'''

with open("/kaggle/working/app.py", "w") as f:
    f.write(app_code)
print("app.py written.")

app.py written.


In [32]:
html_code = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Empathetic Chatbot</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=DM+Serif+Display:ital@0;1&family=DM+Sans:wght@300;400;500&display=swap" rel="stylesheet">
<style>
:root{--bg:#0e0f14;--surface:#16181f;--panel:#1c1f28;--border:#2a2d38;--text:#e2e4ef;--muted:#7a7d8e;--accent:#6c8ef5;--accent2:#38c4a8;--user-bg:#1e2236;--bot-bg:#161c26;--radius:14px;}
*{box-sizing:border-box;margin:0;padding:0;}
body{background:var(--bg);color:var(--text);font-family:'DM Sans',sans-serif;height:100vh;display:flex;flex-direction:column;overflow:hidden;}
header{display:flex;align-items:center;justify-content:space-between;padding:16px 24px;background:var(--surface);border-bottom:1px solid var(--border);flex-shrink:0;}
.header-left{display:flex;align-items:center;gap:12px;}
.logo-dot{width:36px;height:36px;border-radius:10px;background:linear-gradient(135deg,var(--accent),var(--accent2));display:flex;align-items:center;justify-content:center;font-size:17px;}
.header-title h1{font-family:'DM Serif Display',serif;font-size:17px;font-weight:400;}
.header-title p{font-size:10px;color:var(--muted);margin-top:1px;text-transform:uppercase;letter-spacing:.05em;}
.header-right{display:flex;align-items:center;gap:10px;}
.status-dot{width:8px;height:8px;border-radius:50%;background:#22c55e;box-shadow:0 0 8px rgba(34,197,94,.6);animation:pulse 2s infinite;}
@keyframes pulse{0%,100%{opacity:1}50%{opacity:.4}}
.status-label{font-size:12px;color:var(--muted);}
.clear-btn{padding:6px 14px;border-radius:8px;border:1px solid var(--border);background:transparent;color:var(--muted);font-size:12px;font-family:'DM Sans',sans-serif;cursor:pointer;transition:all .2s;}
.clear-btn:hover{border-color:var(--accent);color:var(--accent);}
.main{display:flex;flex:1;overflow:hidden;}
.sidebar{width:230px;background:var(--surface);border-right:1px solid var(--border);padding:18px 14px;display:flex;flex-direction:column;gap:18px;flex-shrink:0;overflow-y:auto;}
.sidebar-section h3{font-size:10px;font-weight:500;color:var(--muted);text-transform:uppercase;letter-spacing:.1em;margin-bottom:8px;}
.emotion-card{background:var(--panel);border:1px solid var(--border);border-radius:12px;padding:12px;}
.emotion-name{font-family:'DM Serif Display',serif;font-size:19px;margin-bottom:5px;transition:color .4s;text-transform:capitalize;}
.conf-wrap{background:var(--border);border-radius:4px;height:4px;margin-top:7px;}
.conf-bar{height:4px;border-radius:4px;background:var(--accent);transition:width .5s ease,background .4s;width:0%;}
.conf-lbl{font-size:11px;color:var(--muted);margin-top:4px;}
.stat-row{display:flex;justify-content:space-between;align-items:center;padding:5px 0;border-bottom:1px solid var(--border);font-size:12px;}
.stat-row:last-child{border-bottom:none;}
.stat-label{color:var(--muted);}
.stat-val{font-weight:500;}
.emotion-chip{display:inline-flex;align-items:center;gap:5px;padding:4px 9px;border-radius:20px;font-size:11px;background:var(--panel);border:1px solid var(--border);margin:2px 2px;}
.emotion-chip .dot{width:6px;height:6px;border-radius:50%;}
.chat-wrap{flex:1;display:flex;flex-direction:column;overflow:hidden;}
.messages{flex:1;overflow-y:auto;padding:22px 26px;display:flex;flex-direction:column;gap:18px;scroll-behavior:smooth;}
.messages::-webkit-scrollbar{width:4px;}
.messages::-webkit-scrollbar-thumb{background:var(--border);border-radius:4px;}
.msg-row{display:flex;gap:10px;align-items:flex-start;animation:fadeUp .3s ease forwards;opacity:0;}
@keyframes fadeUp{from{opacity:0;transform:translateY(8px)}to{opacity:1;transform:translateY(0)}}
.msg-row.user{flex-direction:row-reverse;}
.avatar{width:32px;height:32px;border-radius:9px;display:flex;align-items:center;justify-content:center;font-size:14px;flex-shrink:0;}
.user-av{background:linear-gradient(135deg,#3b52a0,#5b6ec4);}
.bot-av{background:linear-gradient(135deg,#1a4a3a,#2a6b55);}
.bubble-wrap{max-width:62%;display:flex;flex-direction:column;gap:4px;}
.msg-row.user .bubble-wrap{align-items:flex-end;}
.bubble{padding:11px 15px;border-radius:var(--radius);font-size:14px;line-height:1.6;word-break:break-word;}
.user .bubble{background:var(--user-bg);border:1px solid #2a3050;border-bottom-right-radius:4px;}
.bot .bubble{background:var(--bot-bg);border:1px solid var(--border);border-bottom-left-radius:4px;}
.emotion-tag{font-size:10px;color:var(--muted);padding:3px 8px;border-radius:20px;background:var(--panel);border:1px solid var(--border);display:inline-flex;align-items:center;gap:5px;}
.emotion-tag .etdot{width:6px;height:6px;border-radius:50%;}
.typing-indicator{display:flex;gap:5px;align-items:center;padding:13px 15px;}
.typing-indicator span{width:7px;height:7px;border-radius:50%;background:var(--muted);animation:bounce 1.2s infinite;}
.typing-indicator span:nth-child(2){animation-delay:.2s;}
.typing-indicator span:nth-child(3){animation-delay:.4s;}
@keyframes bounce{0%,60%,100%{transform:translateY(0)}30%{transform:translateY(-6px)}}
.input-area{padding:14px 26px 18px;background:var(--surface);border-top:1px solid var(--border);flex-shrink:0;}
.input-row{display:flex;gap:10px;align-items:flex-end;background:var(--panel);border:1px solid var(--border);border-radius:15px;padding:9px 12px;transition:border-color .2s;}
.input-row:focus-within{border-color:var(--accent);}
#msgInput{flex:1;background:transparent;border:none;outline:none;color:var(--text);font-family:'DM Sans',sans-serif;font-size:14px;line-height:1.5;resize:none;max-height:100px;min-height:22px;}
#msgInput::placeholder{color:var(--muted);}
.send-btn{width:34px;height:34px;border-radius:9px;border:none;background:var(--accent);color:white;cursor:pointer;display:flex;align-items:center;justify-content:center;flex-shrink:0;transition:all .2s;font-size:15px;}
.send-btn:hover{background:#5a7de8;transform:scale(1.05);}
.send-btn:disabled{opacity:.4;cursor:not-allowed;transform:none;}
.input-hint{font-size:11px;color:var(--muted);margin-top:7px;text-align:center;}
.welcome{text-align:center;padding:38px 20px;opacity:.75;}
.welcome h2{font-family:'DM Serif Display',serif;font-size:24px;font-weight:400;margin-bottom:9px;}
.welcome p{font-size:14px;color:var(--muted);max-width:340px;margin:0 auto;line-height:1.6;}
.welcome-chips{display:flex;flex-wrap:wrap;gap:7px;justify-content:center;margin-top:18px;}
.welcome-chip{padding:7px 13px;border-radius:20px;border:1px solid var(--border);background:var(--panel);font-size:12px;color:var(--muted);cursor:pointer;transition:all .2s;}
.welcome-chip:hover{border-color:var(--accent);color:var(--accent);}
</style>
</head>
<body>
<header>
  <div class="header-left">
    <div class="logo-dot">🫶</div>
    <div class="header-title">
      <h1>Empathetic Chatbot</h1>
      <p>RoBERTa · BART · Emotion-Gated Attention</p>
    </div>
  </div>
  <div class="header-right">
    <div class="status-dot" id="statusDot"></div>
    <span class="status-label" id="statusLabel">Loading...</span>
    <button class="clear-btn" onclick="clearChat()">Clear chat</button>
  </div>
</header>
<div class="main">
  <aside class="sidebar">
    <div class="sidebar-section">
      <h3>Current Emotion</h3>
      <div class="emotion-card">
        <div class="emotion-name" id="emotionName" style="color:var(--muted)">—</div>
        <div class="conf-wrap"><div class="conf-bar" id="confBar"></div></div>
        <div class="conf-lbl" id="confLbl">Awaiting input</div>
      </div>
    </div>
    <div class="sidebar-section">
      <h3>Session Stats</h3>
      <div class="stat-row"><span class="stat-label">Messages</span><span class="stat-val" id="sMsgs">0</span></div>
      <div class="stat-row"><span class="stat-label">Emotions detected</span><span class="stat-val" id="sEmotions">0</span></div>
      <div class="stat-row"><span class="stat-label">Fallbacks</span><span class="stat-val" id="sFallbacks">0</span></div>
    </div>
    <div class="sidebar-section">
      <h3>Emotion History</h3>
      <div id="emotionHistory"></div>
    </div>
    <div class="sidebar-section" style="margin-top:auto">
      <h3>Model Info</h3>
      <div class="stat-row"><span class="stat-label">Classifier</span><span class="stat-val">RoBERTa</span></div>
      <div class="stat-row"><span class="stat-label">Generator</span><span class="stat-val">BART-base</span></div>
      <div class="stat-row"><span class="stat-label">BLEU-4</span><span class="stat-val">2.14</span></div>
      <div class="stat-row"><span class="stat-label">Perplexity</span><span class="stat-val">17.42</span></div>
    </div>
  </aside>
  <div class="chat-wrap">
    <div class="messages" id="messages">
      <div class="welcome" id="welcomeMsg">
        <h2>How are you feeling today?</h2>
        <p>I am here to listen and respond with empathy. Share anything on your mind.</p>
        <div class="welcome-chips">
          <div class="welcome-chip" onclick="sendQuick('I feel really overwhelmed today')">Feeling overwhelmed</div>
          <div class="welcome-chip" onclick="sendQuick('I just got some really exciting news!')">Exciting news</div>
          <div class="welcome-chip" onclick="sendQuick('I am feeling a bit lonely lately')">Feeling lonely</div>
          <div class="welcome-chip" onclick="sendQuick('Something happened that made me angry')">Feeling angry</div>
        </div>
      </div>
    </div>
    <div class="input-area">
      <div class="input-row">
        <textarea id="msgInput" rows="1" placeholder="Share what is on your mind..." onkeydown="handleKey(event)" oninput="autoResize(this)"></textarea>
        <button class="send-btn" id="sendBtn" onclick="sendMessage()">&#10148;</button>
      </div>
      <div class="input-hint">Enter to send &nbsp;·&nbsp; Shift+Enter for new line</div>
    </div>
  </div>
</div>
<script>
const COLORS={"sadness":"#5b8dee","grief":"#5b8dee","fear":"#7c5cbf","anger":"#e05555","annoyance":"#e07a55","excitement":"#e8a838","joy":"#f0c040","admiration":"#e8a838","surprise":"#38c4a8","curiosity":"#38a8c4","confusion":"#8888cc","love":"#e05580","gratitude":"#55c460","optimism":"#66bb6a","neutral":"#888899","caring":"#55b4c4"};
let msgCount=0,emotionCount=0,fallbackCount=0,emotionLog=[];
async function checkStatus(){
  try{const r=await fetch("/status");const d=await r.json();
    if(d.model_loaded){document.getElementById("statusDot").style.background="#22c55e";document.getElementById("statusLabel").textContent="Ready · "+d.device.toUpperCase();}
    else{document.getElementById("statusDot").style.background="#ef4444";document.getElementById("statusLabel").textContent="Model not loaded";}
  }catch(e){document.getElementById("statusLabel").textContent="Offline";}
}
checkStatus();
function autoResize(el){el.style.height="auto";el.style.height=Math.min(el.scrollHeight,100)+"px";}
function handleKey(e){if(e.key==="Enter"&&!e.shiftKey){e.preventDefault();sendMessage();}}
function sendQuick(t){document.getElementById("msgInput").value=t;sendMessage();}
function appendMsg(text,role,ed){
  const w=document.getElementById("welcomeMsg");if(w)w.remove();
  const msgs=document.getElementById("messages");
  const row=document.createElement("div");row.className="msg-row "+role;
  const av=document.createElement("div");av.className="avatar "+(role==="user"?"user-av":"bot-av");av.textContent=role==="user"?"👤":"🤖";
  const bw=document.createElement("div");bw.className="bubble-wrap";
  const b=document.createElement("div");b.className="bubble";b.textContent=text;bw.appendChild(b);
  if(ed&&role==="bot"){const t=document.createElement("div");t.className="emotion-tag";const c=COLORS[ed.emotion_raw]||"#888899";t.innerHTML=`<span class="etdot" style="background:${c}"></span>${ed.emotion} · ${ed.confidence}%`;bw.appendChild(t);}
  row.appendChild(av);row.appendChild(bw);msgs.appendChild(row);msgs.scrollTop=msgs.scrollHeight;
}
function showTyping(){const msgs=document.getElementById("messages");const row=document.createElement("div");row.className="msg-row bot";row.id="typingRow";const av=document.createElement("div");av.className="avatar bot-av";av.textContent="🤖";const bw=document.createElement("div");bw.className="bubble-wrap";const b=document.createElement("div");b.className="bubble";b.innerHTML='<div class="typing-indicator"><span></span><span></span><span></span></div>';bw.appendChild(b);row.appendChild(av);row.appendChild(bw);msgs.appendChild(row);msgs.scrollTop=msgs.scrollHeight;}
function removeTyping(){const t=document.getElementById("typingRow");if(t)t.remove();}
function updateSidebar(d){
  const c=COLORS[d.emotion_raw]||"#888899";
  document.getElementById("emotionName").textContent=d.emotion;document.getElementById("emotionName").style.color=c;
  document.getElementById("confBar").style.width=d.confidence+"%";document.getElementById("confBar").style.background=c;
  document.getElementById("confLbl").textContent="Confidence: "+d.confidence+"%";
  emotionCount++;if(d.is_fallback)fallbackCount++;emotionLog.push({e:d.emotion_raw,c});
  document.getElementById("sMsgs").textContent=msgCount;document.getElementById("sEmotions").textContent=emotionCount;document.getElementById("sFallbacks").textContent=fallbackCount;
  const h=document.getElementById("emotionHistory");h.innerHTML="";
  emotionLog.slice(-6).reverse().forEach(x=>{const chip=document.createElement("div");chip.className="emotion-chip";chip.innerHTML=`<span class="dot" style="background:${x.c}"></span>${x.e}`;h.appendChild(chip);});
}
async function sendMessage(){
  const inp=document.getElementById("msgInput");const btn=document.getElementById("sendBtn");
  const text=inp.value.trim();if(!text)return;
  inp.value="";inp.style.height="auto";btn.disabled=true;msgCount++;
  appendMsg(text,"user",null);showTyping();
  try{
    const res=await fetch("/chat",{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({message:text})});
    const data=await res.json();removeTyping();
    if(data.error){appendMsg("Sorry, something went wrong.","bot",null);}
    else{appendMsg(data.response,"bot",data);updateSidebar(data);}
  }catch(e){removeTyping();appendMsg("Connection error.","bot",null);}
  btn.disabled=false;inp.focus();
}
async function clearChat(){
  await fetch("/clear",{method:"POST"});
  const msgs=document.getElementById("messages");
  msgs.innerHTML=`<div class="welcome" id="welcomeMsg"><h2>How are you feeling today?</h2><p>I am here to listen and respond with empathy.</p><div class="welcome-chips"><div class="welcome-chip" onclick="sendQuick('I feel really overwhelmed today')">Feeling overwhelmed</div><div class="welcome-chip" onclick="sendQuick('I just got some really exciting news!')">Exciting news</div><div class="welcome-chip" onclick="sendQuick('I am feeling a bit lonely lately')">Feeling lonely</div><div class="welcome-chip" onclick="sendQuick('Something happened that made me angry')">Feeling angry</div></div></div>`;
  msgCount=0;emotionCount=0;fallbackCount=0;emotionLog=[];
  ["sMsgs","sEmotions","sFallbacks"].forEach(id=>document.getElementById(id).textContent="0");
  document.getElementById("emotionHistory").innerHTML="";
  document.getElementById("emotionName").textContent="—";document.getElementById("emotionName").style.color="var(--muted)";
  document.getElementById("confBar").style.width="0%";document.getElementById("confLbl").textContent="Awaiting input";
}
</script>
</body>
</html>'''

with open("/kaggle/working/index.html", "w") as f:
    f.write(html_code)
print("index.html written to /kaggle/working/index.html")

index.html written to /kaggle/working/index.html


## Chatbot Frontend Interface

In [38]:
import os, time, subprocess
from pyngrok import ngrok, conf

# Kill everything old
os.system("pkill -f 'python app.py' 2>/dev/null || true")
os.system("pkill -f ngrok 2>/dev/null || true")
os.system("fuser -k 5000/tcp 2>/dev/null || true")
time.sleep(4)
print("Cleaned up.")

# Start Flask
flask_proc = subprocess.Popen(
    ["python", "/kaggle/working/app.py"], 
    stdout=open("/kaggle/working/flask.log", "w"),
    stderr=subprocess.STDOUT
)

# Wait for model to load and Flask to be ready
print("Waiting for Flask + model to load ", end="")
for i in range(120):
    time.sleep(2)
    try:
        log = open("/kaggle/working/flask.log").read()
        if "Serving Flask app" in log:
            print(" ✅ Flask ready!")
            break
    except:
        pass
    print(".", end="", flush=True)
else:
    print("\n❌ Flask failed. Log:")
    os.system("cat /kaggle/working/flask.log")
    raise RuntimeError("Flask failed to start")

# Connect ngrok AFTER Flask is confirmed up
try:
    ngrok.kill()
    time.sleep(2)
except:
    pass

conf.get_default().auth_token = "3CeWT5SSrcwn9qyPIBcXTmB3OSL_2zPNKQu25k3e41zEnK6AC"
public_url = ngrok.connect(5000)
print(f"\n✅ App is live at: {public_url}")

Cleaned up.
Waiting for Flask + model to load ...... ✅ Flask ready!

✅ App is live at: NgrokTunnel: "https://basin-crane-morphine.ngrok-free.dev" -> "http://localhost:5000"


## Backend Chat Simulation (Kaggle Console)

In [58]:
import torch
from transformers.modeling_outputs import BaseModelOutput
from config import MAX_LEN, DEVICE, LABEL_TO_EMOTION, CONFIDENCE_THRESHOLD, NEUTRAL_IDX, GO_EMOTIONS

EMOTION_TO_ED_TAG = {
    'admiration':    'impressed', 'amusement':     'amusing', 'anger':         'furious',
    'annoyance':     'annoyed', 'approval':      'content', 'caring':        'caring',
    'confusion':     'confused', 'curiosity':     'curious', 'desire':        'hopeful',
    'disappointment':'disappointed', 'disapproval':   'disappointed', 'disgust':       'disgusted',
    'embarrassment': 'embarrassed', 'excitement':    'excited', 'fear':          'afraid',
    'gratitude':     'grateful', 'grief':         'devastated', 'joy':           'joyful',
    'love':          'sentimental', 'nervousness':   'anxious', 'optimism':      'hopeful',
    'pride':         'proud', 'realization':   'surprised', 'relief':        'relieved',
    'remorse':       'guilty', 'sadness':       'sad', 'surprise':      'surprised', 'neutral': 'neutral'
}

EMOTION_MERGE = {
    'grief': 'sadness', 'remorse': 'sadness', 'disappointment': 'sadness', 'annoyance': 'anger',
    'disgust': 'anger', 'disapproval': 'anger', 'nervousness': 'fear', 'realization': 'surprise',
    'relief': 'optimism', 'desire': 'optimism'
}

RESPONSE_TONE_MAP = {
    'sadness':      'caring',
    'anger':        'caring',
    'fear':         'caring',
    'embarrassment':'caring',
    'confusion':    'curious',
    'optimism':     'hopeful',
    'excitement':   'excited',
    'joy':          'joyful',
    'surprise':     'surprised',
    'pride':        'impressed',
    'admiration':   'impressed',
    'gratitude':    'grateful',
    'love':         'sentimental',
    'curiosity':    'curious',
    'neutral':      'caring',  
}

def custom_chat(model, rob_tokenizer, bart_tokenizer):
    print("\n--- Decoupled Empathetic Chatbot (Logit-Level Pronoun Ban) ---")
    print("Type 'quit' or 'exit' to stop.")
    print("Type 'clear' to wipe memory for independent testing.\n")
    
    # ── THE ULTIMATE PROPER FIX ──
    # We strip all first-person pronouns directly from the tokenizer's vocabulary
    forbidden_words = [
        "I", " I", "i", " i", "I'm", " I'm", "i'm", " i'm", 
        "me", " me", "my", " my", "mine", " mine", "myself", " myself", 
        "I'll", " I'll", "i'll", " i'll", "I've", " I've", "i've", " i've",
        "We", " We", "we", " we", "Us", " Us", "us", " us",
        "I'd", " I'd", "i'd", " i'd"
    ]
    
    bad_words_ids = []
    for word in forbidden_words:
        ids = bart_tokenizer(word, add_special_tokens=False).input_ids
        bad_words_ids.append(ids)

    conversation_history = []

    with torch.no_grad():
        while True:
            try:
                user_input = input("You: ")
                if user_input.lower() in ['quit', 'exit']: break
                
                if user_input.lower() == 'clear':
                    conversation_history = []
                    print("[Memory wiped. Starting fresh conversation.]\n")
                    continue

                # ── Step 1: Detect emotion via RoBERTa (Phase 1) ───────────
                rob_inputs = rob_tokenizer(user_input, max_length=MAX_LEN, truncation=True, return_tensors='pt').to(DEVICE)
                rob_out = model.roberta(input_ids=rob_inputs['input_ids'], attention_mask=rob_inputs['attention_mask'])
                
                cls    = rob_out.last_hidden_state[:, 0, :]
                probs  = torch.softmax(model.classifier(cls), dim=-1)
                max_prob, pred_idx = torch.max(probs, dim=-1)

                prob        = max_prob.item()
                raw_emotion = LABEL_TO_EMOTION[pred_idx.item()]
                emotion     = EMOTION_MERGE.get(raw_emotion, raw_emotion)

                if prob < CONFIDENCE_THRESHOLD:
                    emotion = 'neutral'
                    probs[0] = 0.0
                    probs[0, GO_EMOTIONS.index('neutral')] = 1.0
                    print(f"*[Low confidence ({prob:.2f}) → neutral]*")
                else:
                    label = raw_emotion if raw_emotion == emotion else f"{raw_emotion} → {emotion}"
                    print(f"*[Emotion: {label} (Conf: {prob:.2f})]*")

                # ── Step 2: Build Context & Tone (Phase 2 Preparation) ────────
                response_emotion = RESPONSE_TONE_MAP.get(emotion, 'caring')
                
                conversation_history.append(f"User: {user_input}")
                history_context = " </s> ".join(conversation_history[-6:])
                full_context = f"[{response_emotion}] {history_context}"

                bart_inputs = bart_tokenizer(full_context, max_length=MAX_LEN, truncation=True, return_tensors='pt').to(DEVICE)

                # ── Step 3: FastText Fusion & Generation ──────────────
                e_emotion = torch.matmul(probs, model.emotion_embeddings.weight).unsqueeze(1)
                enc_out = model.bart.model.encoder(input_ids=bart_inputs['input_ids'], attention_mask=bart_inputs['attention_mask'], return_dict=True)
                
                H_fused = model.fusion(enc_out.last_hidden_state, e_emotion)
                encoder_outputs = BaseModelOutput(last_hidden_state=H_fused, hidden_states=enc_out.hidden_states, attentions=enc_out.attentions)

                # Generate with forced bad_words_ids (Goodbye Persona Drift!)
                gen_ids = model.bart.generate(
                    encoder_outputs=encoder_outputs,
                    attention_mask=bart_inputs['attention_mask'],
                    max_new_tokens=40,           
                    min_length=10,               
                    no_repeat_ngram_size=3,
                    do_sample=True,          
                    top_p=0.92, 
                    temperature=0.80,            
                    num_beams=3,             
                    num_return_sequences=1,
                    bad_words_ids=bad_words_ids  # <-- This mathematically blocks "I", "me", "my"
                )

                response = bart_tokenizer.decode(gen_ids[0], skip_special_tokens=True).strip()
                print(f"Bot: {response}\n")

                conversation_history.append(f"Bot: {response}")
                if len(conversation_history) > 12:
                    conversation_history = conversation_history[-12:]

            except KeyboardInterrupt:
                break

# Execute Inference Loop
custom_chat(model, rob_tokenizer, bart_tokenizer)


--- Decoupled Empathetic Chatbot (Logit-Level Pronoun Ban) ---
Type 'quit' or 'exit' to stop.
Type 'clear' to wipe memory for independent testing.



You:  I've been studying for hours every single day but I still feel unprepared.


*[Emotion: disappointment → sadness (Conf: 0.67)]*
Bot: That's terrible. What are you going to do?



You:  I think I'm just going to take a break and watch a movie to calm down. 


*[Emotion: neutral (Conf: 0.54)]*
Bot: That sounds like a good idea.



...

You:  The weather has been pretty normal this week


*[Emotion: admiration (Conf: 0.40)]*
Bot: That's good. What's going on?



You:  clear


[Memory wiped. Starting fresh conversation.]



You:  quit


## BLEU Score

In [59]:
import sacrebleu, torch
from datasets import load_dataset
from tqdm import tqdm
from config import MAX_LEN, DEVICE

def compute_bleu(model, rob_tokenizer, bart_tokenizer, n_samples=500):
    raw = load_dataset("empathetic_dialogues", trust_remote_code=True)

    conversations = {}
    for row in raw["validation"]:
        conversations.setdefault(row["conv_id"], []).append(row)

    val_pairs = []
    for turns in conversations.values():
        # Grab the gold emotion for this conversation
        conv_emotion = turns[0].get('context', 'neutral').strip().lower()
        if not conv_emotion or len(conv_emotion.split()) > 2:
            conv_emotion = 'neutral'

        history = []
        for i in range(len(turns) - 1):
            user_text = turns[i]["utterance"].replace("_comma_", ",")
            bot_text  = turns[i+1]["utterance"].replace("_comma_", ",")
            history.append(f"User: {user_text}")
            
            # Prepend the emotion tag exactly like training
            history_str = " </s> ".join(history[-6:])
            full_context = f"[{conv_emotion}] {history_str}"
            
            val_pairs.append({
                "context":          full_context,
                "latest_utterance": user_text,
                "reference":        bot_text,
            })
            history.append(f"Bot: {bot_text}")

    val_pairs = val_pairs[:n_samples]
    hypotheses, references = [], []

    model.eval()
    with torch.no_grad():
        for item in tqdm(val_pairs, desc="BLEU"):
            # For BLEU, we just want standard generation, identical to inference.py
            rob_in  = rob_tokenizer(item["latest_utterance"],
                        max_length=MAX_LEN, truncation=True, return_tensors="pt").to(DEVICE)
            bart_in = bart_tokenizer(item["context"],
                        max_length=MAX_LEN, truncation=True, return_tensors="pt").to(DEVICE)

            # --- ADDED: Matching the robust generation parameters from inference.py ---
            gen_ids, _, _ = model.generate_response(
                rob_input_ids=rob_in["input_ids"],
                rob_attention_mask=rob_in["attention_mask"],
                bart_input_ids=bart_in["input_ids"],
                bart_attention_mask=bart_in["attention_mask"],
                max_new_tokens=60, 
                min_length=15,          
                num_beams=5,
                length_penalty=1.5,     
                early_stopping=True, 
                no_repeat_ngram_size=3,
            )
            
            hypotheses.append(bart_tokenizer.decode(gen_ids[0], skip_special_tokens=True).strip())
            references.append(item["reference"])

    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    print(f"\nBLEU-4 : {bleu.score:.2f}")
    print(f"BP     : {bleu.bp:.4f}  (1.0 = no brevity penalty)")
    
    # --- ADDED: Print 5 side-by-side examples so you can see why BP/BLEU behave this way! ---
    print(f"\n{'='*50}")
    print("SAMPLE PREDICTIONS vs REFERENCES")
    print(f"{'='*50}")
    for i in range(5):
        print(f"User Said : {val_pairs[i]['latest_utterance']}")
        print(f"Reference : {references[i]}")
        print(f"Prediction: {hypotheses[i]}")
        print("-" * 50)
        
    return bleu, hypotheses, references

bleu_result, hyps, refs = compute_bleu(model, rob_tokenizer, bart_tokenizer, n_samples=500)


Generating train split:   0%|          | 0/76673 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/12030 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10943 [00:00<?, ? examples/s]

BLEU: 100%|██████████| 500/500 [01:36<00:00,  5.17it/s]


BLEU-4 : 2.14
BP     : 0.9109  (1.0 = no brevity penalty)

SAMPLE PREDICTIONS vs REFERENCES
User Said : Today,as i was leaving for work in the morning,i had a tire burst in the middle of a busy road. That scared the hell out of me!
Reference : Are you fine now?
Prediction: Oh no! That must have been so scary! Did you get out of the car?
--------------------------------------------------
User Said : Are you fine now?
Reference : Yeah,i'm doing alright now, but with minor injuries.
Prediction: I am fine, but i was so scared that i was going to die.
--------------------------------------------------
User Said : Yeah,i'm doing alright now, but with minor injuries.
Reference : Cool :) Is your car damaged a lot?
Prediction: That's good to hear. I'm glad you're okay now.
--------------------------------------------------
User Said : Cool :) Is your car damaged a lot?
Reference : The car was badly damaged,i veered outside the road and hit a tree trunk. next thing is insurance follow up.
Predi

## Perplexity Score Evaluation

In [60]:
import math, torch
from datasets import load_dataset
from tqdm import tqdm
from config import MAX_LEN, DEVICE

def compute_perplexity(model, rob_tokenizer, bart_tokenizer, n_samples=500):
    raw = load_dataset("empathetic_dialogues", trust_remote_code=True)

    conversations = {}
    for row in raw["validation"]:
        conversations.setdefault(row["conv_id"], []).append(row)

    val_pairs = []
    for turns in conversations.values():
        conv_emotion = turns[0].get('context', 'neutral').strip().lower()
        if not conv_emotion or len(conv_emotion.split()) > 2:
            conv_emotion = 'neutral'

        history = []
        for i in range(len(turns) - 1):
            user_text = turns[i]["utterance"].replace("_comma_", ",")
            bot_text  = turns[i+1]["utterance"].replace("_comma_", ",")
            history.append(f"User: {user_text}")
            
            history_str = " </s> ".join(history[-6:])
            full_context = f"[{conv_emotion}] {history_str}"
            
            val_pairs.append({
                "context":          full_context,
                "latest_utterance": user_text,
                "reference":        bot_text,
            })
            history.append(f"Bot: {bot_text}")

    val_pairs = val_pairs[:n_samples]
    total_loss, total_tokens = 0.0, 0

    model.eval()
    with torch.no_grad():
        for item in tqdm(val_pairs, desc="Perplexity"):
            rob_in  = rob_tokenizer(item["latest_utterance"],
                        max_length=MAX_LEN, truncation=True,
                        padding="max_length", return_tensors="pt").to(DEVICE)
            bart_in = bart_tokenizer(item["context"],
                        max_length=MAX_LEN, truncation=True,
                        padding="max_length", return_tensors="pt").to(DEVICE)
            target  = bart_tokenizer(item["reference"],
                        max_length=MAX_LEN, truncation=True,
                        padding="max_length", return_tensors="pt").to(DEVICE)

            target_ids = target["input_ids"].clone()
            target_ids[target_ids == bart_tokenizer.pad_token_id] = -100

            outputs = model.forward_phase2(
                rob_input_ids=rob_in["input_ids"],
                rob_attention_mask=rob_in["attention_mask"],
                bart_input_ids=bart_in["input_ids"],
                bart_attention_mask=bart_in["attention_mask"],
                labels=target_ids,
            )
            n_real = (target_ids != -100).sum().item()
            if n_real > 0:
                total_loss   += outputs["loss_gen"].item() * n_real
                total_tokens += n_real

    ppl = math.exp(total_loss / total_tokens)
    print(f"\nPerplexity : {ppl:.2f}  (target: below 35)")
    return ppl

ppl = compute_perplexity(model, rob_tokenizer, bart_tokenizer, n_samples=500)


Perplexity: 100%|██████████| 500/500 [00:13<00:00, 36.16it/s]


Perplexity : 17.42  (target: below 35)


In [61]:
# Verify the two code paths produce identical encoder inputs
# (chat cell vs generate_response in model.py)
import torch
from config import MAX_LEN, DEVICE

test = "I just got accepted into my dream college!"
ed_tag = "excited"
context = f"[{ed_tag}] User: {test}"

# Path 1: what the chat cell does
bart_in_chat = bart_tokenizer(context, max_length=MAX_LEN, truncation=True, return_tensors='pt').to(DEVICE)

# Path 2: what generate_response receives
bart_in_gen = bart_tokenizer(context, max_length=MAX_LEN, truncation=True, return_tensors='pt').to(DEVICE)

match = torch.equal(bart_in_chat['input_ids'], bart_in_gen['input_ids'])
print(f"Both paths produce identical input_ids: {match}")
print(f"First 10 tokens: {bart_in_chat['input_ids'][0][:10]}")
print(f"Decoded prefix: {bart_tokenizer.decode(bart_in_chat['input_ids'][0][:6])}")
# Should show: [excited] User: I just...

Both paths produce identical input_ids: True
First 10 tokens: tensor([    0, 10975, 34645,  4560,   742, 27913,    35,    38,    95,   300],
       device='cuda:0')
Decoded prefix: <s>[excited] User
